In [1]:
import os

import pandas as pd
import rasterio
import geopandas as gpd
import numpy as np
from shapely.geometry import box
from rasterio import features


In [2]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf, extract_sample, get_window

In [3]:
lc_tiles_dir = '/home/jan/Documents/EPFL/project_2/data/input_datasets/worldcover'
lc_tiles = get_tiles_as_gdf(lc_tiles_dir)

cocoa_ecuador = gpd.read_file('../data/tmp/cacao_ecuador.gpkg')
cocoa_ecuador = cocoa_ecuador.to_crs(4326)

In [4]:
dataset_extent = box(*cocoa_ecuador.total_bounds)

In [5]:
lc_tiles_filtered = lc_tiles.loc[lc_tiles.intersects(dataset_extent)]

In [8]:
out_dir = '../data/input_datasets/cocoa_ecuador'
for idx, row in lc_tiles_filtered.iterrows():
    print(idx)
    cocoa_ecuador_in_tile = cocoa_ecuador.loc[cocoa_ecuador.intersects(row.geometry)]
    
    with rasterio.open(row.filename) as src:
        lc_map = src.read(1)
        profile = src.profile

        shapes = [(geom, 1) for geom in cocoa_ecuador_in_tile.geometry]
        if not shapes:
            continue
        # Rasterize the shapes
        rasterized_cocoa = features.rasterize(shapes, out_shape=(profile['height'], profile['width']),
                               transform=profile['transform'], fill=0, all_touched=True, dtype='uint8')

    # Conditions
    is_forest_cover = lc_map == 10
    is_cocoa = rasterized_cocoa == 1
    cond = is_forest_cover & is_cocoa
    # New array based on conditions
    coco_filtered = np.where(cond, 1, 0) 

    out_filename = os.path.join(out_dir, row.filename.split('/')[-1].split('_')[-2] + '.tif')
    
    #Save the rasterized shapes to a new file
    with rasterio.open(out_filename, 'w', **profile) as dst:
        dst.write(coco_filtered, 1)

432
433
618
619
638
639
